In [1]:
!pip install pyspark -q
!pip install boto3 -q
!pip install pyarrow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")
else:
    print("✅ Drive already mounted, skipping...")

Mounted at /content/drive
✅ Drive mounted successfully


In [3]:
#configuration notebook

import os
os.chdir("/content/drive/My Drive/Colab Notebooks")
%run oo_config.ipynb

In [4]:
import requests
import boto3
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import glob
from datetime import datetime as dt

# ── Session ───────────────────────────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("MyEarthQuakeApp") \
    .getOrCreate()

print("Spark version:", spark.version)

# ── Fetch API ─────────────────────────────────────────────────────────────────
response = requests.get("https://api.data.gov.my/weather/warning/earthquake")

# ── Parse JSON → Spark DataFrame ─────────────────────────────────────────────
rdd = spark.sparkContext.parallelize([response.text])
df = spark.read.json(rdd)

df.printSchema()

# ── Transform ─────────────────────────────────────────────────────────────────
df = df.withColumn(
    "utcdatetime_ts",
    F.expr("try_to_timestamp(utcdatetime, \"yyyy-MM-dd'T'HH:mm:ss\")")
)

df = df.withColumn("year",  F.year("utcdatetime_ts")) \
       .withColumn("month", F.month("utcdatetime_ts"))

# Apply current month filter (COMMENT out to get all)
df = df.filter((F.col("year") == dt.now().year) & (F.col("month") == dt.now().month))

df.orderBy(F.desc("utcdatetime_ts")).show(10)



# ── Write Parquet to dedicated subfolder ─────────────────────────────────────
local_dir  = "/tmp/earthquake"                          # subfolder, not root /tmp
#local_path = os.path.join(local_dir, "earthquake_data.parquet")

os.makedirs(local_dir, exist_ok=True)                   # create if not exists

df.repartition("year", "month") \
  .write \
  .mode("overwrite") \
  .partitionBy("year", "month") \
  .option("compression", "snappy") \
  .parquet(local_dir)

# ── Rename part files to deterministic names per partition ────────────────────
parquet_files = []  # list of (local_path, s3_key) tuples

partition_dirs = glob.glob(os.path.join(local_dir, "year=*/month=*/"))

for partition_dir in sorted(partition_dirs):
    part_files = sorted(glob.glob(os.path.join(partition_dir, "part-*.parquet")))

    for idx, old_path in enumerate(part_files):
        # e.g. data_part_000.parquet, data_part_001.parquet, ...
        fixed_name  = f"data_part_{idx:03d}.parquet"
        fixed_path  = os.path.join(partition_dir, fixed_name)
        os.rename(old_path, fixed_path)

        # Preserve partition folder structure for S3 key
        relative    = os.path.relpath(fixed_path, local_dir)
        s3_key      = f"earthquake/{relative}"

        parquet_files.append((fixed_path, s3_key))


# ── Find all .parquet part files across all year/month partitions ─────────────
parquet_files = glob.glob(os.path.join(local_dir, "**/*.parquet"), recursive=True)

# ── Upload to MinIO via boto3 ─────────────────────────────────────────────────
s3 = boto3.client(
    "s3",
    endpoint_url=G_MINIO_ENDPOINT,
    aws_access_key_id=G_MINIO_ACCESS_KEY,
    aws_secret_access_key=G_MINIO_SECRET_KEY
)

try:
    for full_local_path in parquet_files:
        # Preserve partition folder structure: year=2023/month=1/part-xxx.parquet
        relative_path = os.path.relpath(full_local_path, local_dir)
        s3_key = f"earthquake/{relative_path}"

        s3.upload_file(
            Filename=full_local_path,
            Bucket="rawdatasets",
            Key=s3_key
        )
        print(f"Uploaded: {s3_key}")

    print(f"All {len(parquet_files)} file(s) uploaded to MinIO successfully!")
finally:
    # ── Cleanup: only delete /tmp/earthquake subfolder ────────────────────────
    shutil.rmtree(local_dir, ignore_errors=True)
    print(f"Deleted temp directory: {local_dir}")

Spark version: 4.0.2
root
 |-- depth: double (nullable = true)
 |-- lat: double (nullable = true)
 |-- lat_vector: string (nullable = true)
 |-- localdatetime: string (nullable = true)
 |-- location: string (nullable = true)
 |-- location_original: string (nullable = true)
 |-- lon: double (nullable = true)
 |-- lon_vector: string (nullable = true)
 |-- magdefault: double (nullable = true)
 |-- magtypedefault: string (nullable = true)
 |-- n_distancemas: string (nullable = true)
 |-- n_distancerest: string (nullable = true)
 |-- nbm_distancemas: string (nullable = true)
 |-- nbm_distancerest: string (nullable = true)
 |-- status: string (nullable = true)
 |-- utcdatetime: string (nullable = true)
 |-- visible: boolean (nullable = true)

+-----+---------+----------+-------------------+--------------------+--------------------+----------+-----------+----------+--------------+--------------------+--------------------+--------------------+--------------------+------+-------------------+---

In [5]:
spark.stop()